# 25 — Error Handling in NLP
**Goal:** Build robust NLP pipelines that handle real-world messy data.

Every chapter so far assumed clean text. Real resumes are not clean: empty files, binary payloads, wrong encodings, all-caps walls, and stray control characters arrive from PDF/DOCX extractors and email attachments. This chapter is the reality check — it enumerates the failure modes, then builds decoding, fallback, and validation layers that let a pipeline *degrade gracefully* instead of crash.

**Why it matters for resumes / ATS:** a parser that throws on one malformed resume fails the whole batch — or worse, silently writes garbage into the candidate record. Defensive coding here is the difference between a demo and a product: the layers built in this chapter (safe decode → fallback NLP → validate → flag) are exactly what Ch. 26–27's PDF/DOCX pipelines will need, since document extractors are the messiest input source of all.

## 1. Common Resume Parsing Failures

Before defending, enumerate the enemy. Each failure mode here is a real artifact of resume ingestion: empty documents (scanned-but-blank pages), binary fragments (wrong file type opened as text), non-UTF-8 encodings (latin-1 "José" mangled by a Windows extractor), all-caps text (poorly OCR'd headers), missing section structure, and punctuation-only noise.

**What the code does:** builds `error_cases`, a list of `(name, text)` pairs, and prints each with `repr(text[:30])`. The `repr` matters: it shows escape sequences literally, so `"\x00\x01\x02\x03"` prints as backslash-escapes rather than invisible control characters, and `"Jos\xe9"` reveals the latin-1 byte that UTF-8 decoding will reject. The cell only *surveys* the cases; §2–5 build the defenses.

**Try it:** append your own worst case — a resume with an emoji, a lone `\xff` byte, or a 1 MB single line — and watch each later handler deal with it.

In [ ]:
error_cases = [
    ("Empty document", ""),
    ("Binary file", "\x00\x01\x02\x03"),
    ("Encoding issue", "Jos\xe9's r\xe9sum\xe9 — Sr. ML Engr."),
    ("All caps", "PROFESSIONAL SUMMARY: DATA SCIENTIST"),
    ("No sections", "Just a wall of text with no breaks"),
    ("Special chars only", "!!!@@@###$$$%%%%"),
]

print("Common error cases in resume parsing:")
for case_name, text in error_cases:
    print(f"  {case_name:25s} → {repr(text[:30])}...")

## 2. Safe Encoding Handler

Text arrives as bytes, and bytes have no encoding until you say so. UTF-8 is the sane default, but real documents arrive in latin-1, cp1252, or worse. The fix is a **fallback chain**: try the standard, detect on failure, degrade to lossy as a last resort.

**What the code does:** `safe_decode(data)` implements the chain:
1. empty bytes → `""` (no error, no work);
2. `data.decode("utf-8")` — the happy path;
3. on `UnicodeDecodeError`, `chardet.detect(data)` guesses the encoding and prints a warning with its **confidence**;
4. decode with the detected encoding;
5. if even that fails, `decode("utf-8", errors="ignore")` — lossy but never crashing.

Reference run on `b"Jos\xe9's r\xe9sum\xe9"`: chardet guessed `cp1250` at confidence **0.08** (low — short strings give a detector almost nothing to work with), and the function still returned `"José's résumé"` correctly, since cp1250 maps byte `0xE9` to `é`. The lesson: detection is a guess, so always keep the `errors="ignore"` safety net.

In [ ]:
import chardet

def safe_decode(data: bytes) -> str:
    """Handle encoding detection and fallback."""
    if not data:
        return ""
    try:
        # Try UTF-8 first
        return data.decode("utf-8")
    except UnicodeDecodeError:
        # Detect encoding
        detected = chardet.detect(data)
        enc = detected.get("encoding", "latin-1")
        print(f"  Warning: UTF-8 failed, detected {enc} (confidence: {detected['confidence']})")
        try:
            return data.decode(enc)
        except:
            # Fallback: ignore errors
            return data.decode("utf-8", errors="ignore")

# Test
test_data = b"Jos\xe9's r\xe9sum\xe9"
print(f"Raw bytes: {test_data}")
print(f"Decoded:   {safe_decode(test_data)}")

## 3. Graceful NLP Pipeline with Fallbacks

Model failures are a second failure class: the spaCy model may be missing, or a pathological input may make it produce nothing usable. `RobustNLPPipeline` wraps every step so the pipeline keeps producing *something*.

**What the code does:** in `__init__`, `spacy.load("en_core_web_sm")` runs inside `try/except OSError` — a missing model means `spacy_ok=False` and regex-only mode. `extract_entities` then:
- returns `[]` for blank/whitespace input;
- runs spaCy on `text[:100000]` (a hard cap against OOM on huge documents), catching any exception;
- **only if spaCy returned nothing**, falls back to regex patterns for EMAIL / URL / PHONE.

The reference run exposes the design's blind spot: on `"Contact: john@email.com, Phone: +1-555-1234"` the small spaCy model produced `[('Phone', 'ORG'), ('+1-555-1234', 'NORP')]` — wrong labels, and it missed the email entirely — but because the entity list was *non-empty*, the regex fallback never ran. A production version should merge and validate rather than trust the first non-empty result.

In [ ]:
import spacy, re
from typing import Optional, List

class RobustNLPPipeline:
    """NLP pipeline with fallback for every failure mode."""

    def __init__(self):
        try:
            self.nlp = spacy.load("en_core_web_sm")
            self.spacy_ok = True
        except OSError:
            print("Warning: spaCy model not available, using regex-only fallback")
            self.spacy_ok = False

    def extract_entities(self, text: str) -> List[tuple]:
        """Extract entities with fallback."""
        if not text or not text.strip():
            return []

        entities = []

        # Try spaCy first
        if self.spacy_ok:
            try:
                doc = self.nlp(text[:100000])  # Limit to avoid OOM
                entities = [(e.text, e.label_) for e in doc.ents]
            except Exception as e:
                print(f"  Warning: spaCy failed ({e}), falling back to regex")
                entities = []

        # Regex fallback for emails, phones, URLs
        if not entities:
            for pattern, label in [
                (r"[\w.+-]+@[\w-]+\.[\w.]+", "EMAIL"),
                (r"https?://[\w./-]+", "URL"),
                (r"[+]?[\d\s()-]{7,}[\d]", "PHONE"),
            ]:
                for match in re.finditer(pattern, text, re.IGNORECASE):
                    entities.append((match.group(0), label))

        return entities

    def __call__(self, text: str):
        return self.extract_entities(text)

# Test
robust = RobustNLPPipeline()
test_cases = [
    "Contact: john@email.com, Phone: +1-555-1234",
    "",
    "\x00\x01\x02Binary garbage\x03",
]

for tc in test_cases:
    result = robust(tc)
    print(f"Input: {repr(tc[:40]):42s} → Entities: {result}")

## 4. Validation Layer for Extracted Data

Extraction produces strings; the database wants typed, normalized values. Validation is the gate that rejects nonsense before it poisons downstream matching — an email like "not-an-email" or a year like "12" should never reach the profile store.

**What the code does:** three validators, each returning the cleaned value or `None`:
- `validate_email` — strip, lowercase, then a full regex anchored with `^...$`;
- `validate_phone` — keeps only digits and `+`, then requires **7–15** characters;
- `validate_year` — parses to int, accepts only **1950–2030**.

The test loop feeds each input through all three validators and prints the first that accepts it. Reference run: `john@email.com` → valid email; `not-an-email` → invalid; `+1-555-123-4567` → `+15551234567` (13 digits — the test table's masked `+155****4567` is a display label; the function returns the full string); `12` → invalid everywhere; `2020` → year `2020`.

In [ ]:
import re
from typing import Optional

def validate_email(email: str) -> Optional[str]:
    """Validate and normalize email."""
    if not email: return None
    email = email.strip().lower()
    pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    return email if re.match(pattern, email) else None

def validate_phone(phone: str) -> Optional[str]:
    """Validate and normalize phone."""
    if not phone: return None
    digits = re.sub(r"[^\d+]", "", phone)
    return digits if 7 <= len(digits) <= 15 else None

def validate_year(year: str) -> Optional[int]:
    """Validate year is reasonable."""
    try:
        y = int(year.strip())
        return y if 1950 <= y <= 2030 else None
    except: return None

# Test
tests = [
    ("john@email.com", "john@email.com"),
    ("not-an-email", None),
    ("+1-555-123-4567", "+15551234567"),
    ("12", None),
    ("2020", 2020),
]
for raw, expected in tests:
    for validator, name in [(validate_email, "email"), (validate_phone, "phone"), (validate_year, "year")]:
        result = validator(raw)
        if result is not None:
            print(f"  {name:7s} '{raw:20s}' → valid: {result}")
            break
    else:
        print(f"  {'?':7s} '{raw:20s}' → invalid (expected {expected})")

## 5. Empty Section Detection

The last failure mode is structural: a resume with headers but no content — "Skills" followed by nothing. A matcher that reads an empty section as "no skills listed" silently discounts the candidate; flagging it keeps the record honest.

**What the code does:** `detect_empty_sections(text)` splits on `\n(?=[A-Z][A-Za-z /]+\n)` — a lookahead that cuts *before* any line shaped like a section header (capitalized, letters/spaces/slashes). For each part it treats line 0 as the header and counts lines longer than **10 chars** as content; a header with no such lines is reported empty. On the sample resume the reference run flags **Skills, Education, and Certifications** while keeping Professional Summary and Experience (their content lines exceed 10 chars).

**Try it:** the `len(l) > 10` threshold is arbitrary — drop it to 5 and a one-word placeholder like "N/A" suddenly counts as real content. That false-positive/false-negative trade-off is the kind of knob this chapter wants you to notice.

In [ ]:
def detect_empty_sections(text: str) -> List[str]:
    """Find resume sections that exist but have no substantive content."""
    sections = re.split(r"\n(?=[A-Z][A-Za-z /]+\n)", text)
    empty = []
    for section in sections:
        lines = [l.strip() for l in section.split("\n") if l.strip()]
        header = lines[0] if lines else ""
        content_lines = [l for l in lines[1:] if len(l) > 10]
        if header and not content_lines:
            empty.append(header)
    return empty

resume = """Professional Summary
Experienced data scientist.

Skills

Experience
Developed ML models at Google.

Education

Certifications

"""
empty = detect_empty_sections(resume)
print("Empty sections found:")
for section in empty:
    print(f"  ⚠ '{section}' has no content")

## Key Insight: Production NLP needs error handling at every stage. Defensive coding prevents pipeline failures.

**Robustness is a pipeline property, not a library feature — decode safely, fall back, validate, then flag.**

Each layer answers one question: can it crash? (decode: never), can it degrade? (NLP: regex fallback), can it lie? (validators return `None`), can it hide? (empty-section detection flags). The honest caveat from §3 — a fallback that only runs when the primary path returns *nothing* — is the classic production bug. Ch. 26–27 carry these layers into PDF and DOCX parsing, where binary formats and extractor quirks make every one of them mandatory.